# Complejos simpliciales (abstractos)
**Topología Aplicada y Computacional — Tema 1.** Ejercicios de programación (diapositiva 53).

**Convenios**
- Un símplice se representa como un `frozenset` de vértices (hashable → se puede meter en un `set`).
- El complejo se guarda **cerrado bajo caras**: si $\sigma\in K$, todas sus caras no vacías también.
- $\dim(\sigma) = \operatorname{card}(\sigma) - 1$.

> Nota: cada ejercicio añade métodos a la clase con `ComplejoSimplicial.metodo = ...`, un patrón habitual en notebooks para ir construyendo una clase por partes. Ejecuta las celdas en orden.

## Ejercicio 1 — Clase para almacenar complejos simpliciales

In [ ]:
from itertools import combinations


class ComplejoSimplicial:
    def __init__(self, simplices):
        """
        'simplices' es un iterable de símplices. Normalmente se pasan solo los
        símplices MAXIMALES (las caras se generan solas), pero funciona con
        cualquier lista porque siempre cerramos bajo caras.

            K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
        """
        self.simplices = set()
        for s in simplices:
            self._anadir_con_caras(s)

    def _anadir_con_caras(self, s):
        """Añade un símplice y todas sus caras no vacías al complejo."""
        vertices = sorted(frozenset(s))
        for k in range(1, len(vertices) + 1):
            for cara in combinations(vertices, k):
                self.simplices.add(frozenset(cara))

    def __repr__(self):
        return f"ComplejoSimplicial({len(self.simplices)} simplices)"

## Ejercicio 2 — Dimensión del complejo
Máximo de las dimensiones de sus símplices.

In [ ]:
def dimension(self):
    if not self.simplices:
        return -1  # convenio para el complejo vacío
    return max(len(s) - 1 for s in self.simplices)


ComplejoSimplicial.dimension = dimension

## Ejercicios 3 y 4 — Todas las caras y caras de dimensión dada
Todas las caras del complejo son, por definición, sus símplices. La versión por dimensión filtra por cardinal.

In [ ]:
def caras(self):
    return set(self.simplices)


def caras_de_dimension(self, d):
    return {s for s in self.simplices if len(s) - 1 == d}


ComplejoSimplicial.caras = caras
ComplejoSimplicial.caras_de_dimension = caras_de_dimension

## Ejercicios 5 y 6 — Estrella y link de un símplice
$\operatorname{St}(\tau)=\{\sigma\in K \mid \tau\le\sigma\}$ (las cocaras de $\tau$; en general **no** es subcomplejo).

Para el link hace falta la **estrella cerrada** $\overline{\operatorname{St}}(\tau)$ (menor subcomplejo que contiene a $\operatorname{St}(\tau)$), y entonces
$$\operatorname{Lk}(\tau)=\{\sigma\in\overline{\operatorname{St}}(\tau)\mid \sigma\cap\tau=\varnothing\}.$$

In [ ]:
def estrella(self, tau):
    tau = frozenset(tau)
    return {s for s in self.simplices if tau <= s}


def estrella_cerrada(self, tau):
    """Menor subcomplejo que contiene a St(tau): la estrella y todas sus caras."""
    cerrada = set()
    for sigma in self.estrella(tau):
        vertices = sorted(sigma)
        for k in range(1, len(vertices) + 1):
            for cara in combinations(vertices, k):
                cerrada.add(frozenset(cara))
    return cerrada


def link(self, tau):
    tau = frozenset(tau)
    return {s for s in self.estrella_cerrada(tau) if not (s & tau)}


ComplejoSimplicial.estrella = estrella
ComplejoSimplicial.estrella_cerrada = estrella_cerrada
ComplejoSimplicial.link = link

## Utilidad y ejemplo
El complejo de la diapositiva del *poset de caras*: triángulo $\{0,1,2\}$ relleno, arista $\{2,3\}$ y arista $\{3,4\}$.

In [ ]:
def mostrar(conjunto_de_simplices):
    """Imprime un conjunto de símplices de forma legible y ordenada."""
    partes = ["{" + ",".join(map(str, sorted(s))) + "}"
              for s in sorted(conjunto_de_simplices, key=lambda s: (len(s), sorted(s)))]
    return "  ".join(partes) if partes else "(vacio)"


K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
print(K, "| dim =", K.dimension())
print("Todas las caras:\n ", mostrar(K.caras()))

---
# Casos de prueba
Probamos **cada función por separado**, imprimiendo exactamente qué produce y por qué es correcto. Cada celda deja un `assert` de seguridad al final: si algo se rompiera, la celda fallaría; si pasa, la explicación impresa te dice qué ha funcionado.

### Caso 1 — La construcción cierra bajo caras
Pasamos solo los símplices *maximales* y comprobamos que el complejo contiene además **todas** sus caras.

In [ ]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
print("Entrada (maximales): {0,1,2}, {2,3}, {3,4}")
print("Complejo generado ->", len(K.caras()), "símplices:")
print(" ", mostrar(K.caras()))

faltan = [(set(sigma), set(cara))
          for sigma in K.caras()
          for r in range(1, len(sigma))
          for cara in map(frozenset, combinations(sorted(sigma), r))
          if cara not in K.simplices]
assert not faltan
print("\nFunciona: de los 3 símplices maximales se han generado 11, y NO falta ninguna cara.")
print("Por ejemplo, del triángulo {0,1,2} aparecen sus 3 aristas y sus 3 vértices.")

### Caso 2 — Dimensión
La dimensión del complejo es la de su símplice más grande.

In [ ]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
print("dim K =", K.dimension())
assert K.dimension() == 2
print("Funciona: el símplice mayor es el triángulo {0,1,2} (card 3), luego la dimensión es 3-1 = 2.")

### Caso 3 — Caras por dimensión
Cada cara aparece en exactamente un nivel de dimensión; la unión de todos los niveles reconstruye el complejo.

In [ ]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
for d in range(K.dimension() + 1):
    caras_d = K.caras_de_dimension(d)
    print(f"dim {d}: {len(caras_d)} caras ->", mostrar(caras_d))

total = sum(len(K.caras_de_dimension(d)) for d in range(K.dimension() + 1))
assert total == len(K.caras())
print(f"\nFunciona: 5 vértices + 5 aristas + 1 triángulo = {total} = número total de caras.")
print("Los niveles no se solapan ni dejan huecos.")

### Caso 4 — Estrella $\operatorname{St}(\{2\})$
Deben salir justo las cocaras de $\{2\}$ (todo símplice que contiene al vértice 2).

In [ ]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
est = K.estrella({2})
print("St({2}) =", mostrar(est))
esperado = {frozenset(s) for s in [{2}, {0, 2}, {1, 2}, {2, 3}, {0, 1, 2}]}
assert est == esperado
assert all(frozenset({2}) <= s for s in est)
print("Funciona: coincide con la diapositiva y TODO símplice de la estrella contiene al 2.")
print("Nótese que {3,4} no está: no contiene al vértice 2.")

### Caso 5 — Link $\operatorname{Lk}(\{2\})$
El link vive en la estrella cerrada y es disjunto de $\{2\}$.

In [ ]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
lk = K.link({2})
print("Lk({2}) =", mostrar(lk))
esperado = {frozenset(s) for s in [{0}, {1}, {3}, {0, 1}]}
assert lk == esperado
assert all(not (s & frozenset({2})) for s in lk)
print("Funciona: coincide con la diapositiva. Se obtiene quitando el vértice 2 a las cocaras:")
print("  {0,2}->{0}, {1,2}->{1}, {2,3}->{3}, {0,1,2}->{0,1}. Todo disjunto de {2}.")

---
## Casos extremos
Ahora cosas más exigentes: complejo vacío, símplices que no existen, símplices grandes (con su combinatoria), links de caras internas de un tetraedro, el símplice maximal y complejos desconexos.

### Caso 6 — Complejo vacío (nada debe romperse)
Dimensión $-1$ por convenio, y estrella/link de cualquier cosa devuelven el conjunto vacío en lugar de fallar.

In [ ]:
V = ComplejoSimplicial([])
print("caras:", mostrar(V.caras()), "| dim:", V.dimension())
print("St({0}):", mostrar(V.estrella({0})), "| Lk({0}):", mostrar(V.link({0})))
assert V.caras() == set()
assert V.dimension() == -1
assert V.estrella({0}) == set() and V.link({0}) == set()
print("Funciona: el caso vacío se maneja sin errores; dim = -1 y estrella/link vacíos.")

### Caso 7 — Preguntar por un símplice que no está
Robustez: pedir la estrella o el link de algo ausente no lanza excepción.

In [ ]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
print("St({99}) =", mostrar(K.estrella({99})))
print("Lk({99}) =", mostrar(K.link({99})))
assert K.estrella({99}) == set()
assert K.link({99}) == set()
print("Funciona: como el vértice 99 no está en K, no tiene cocaras, y todo sale vacío.")

### Caso 8 — Construcción idempotente
Pasar caras redundantes o repetidas no cambia el complejo: solo importan los conjuntos, no cómo se listen.

In [ ]:
K1 = ComplejoSimplicial([{0, 1, 2}])
K2 = ComplejoSimplicial([{0, 1, 2}, {0, 1}, {1}, {2}, {0, 1, 2}])  # con caras repetidas
assert K1.simplices == K2.simplices
print("Funciona: dar los maximales o darlos con caras repetidas produce EXACTAMENTE el mismo complejo")
print(f"({len(K1.simplices)} símplices en ambos casos).")

### Caso 9 — Combinatoria del símplice (tetraedro sólido)
La diapositiva *Combinatoria del símplice* dice que un $k$-símplice tiene $\binom{k+1}{l+1}$ caras de dimensión $l$ y $2^{k+1}-1$ en total. Lo verificamos con el $3$-símplice.

In [ ]:
from math import comb

T = ComplejoSimplicial([{0, 1, 2, 3}])   # 3-símplice sólido (tetraedro)
k = 3
print("Tetraedro sólido {0,1,2,3}: dim", T.dimension(), "| total símplices", len(T.caras()))
for l in range(k + 1):
    n = len(T.caras_de_dimension(l))
    print(f"  dim {l}: {n} caras   (C(4,{l+1}) = {comb(k + 1, l + 1)})")
    assert n == comb(k + 1, l + 1)
assert len(T.caras()) == 2 ** (k + 1) - 1
print(f"Funciona: 4 vértices, 6 aristas, 4 triángulos, 1 tetraedro = 15 = 2^4 - 1.")
print("Reproduce exactamente la combinatoria del símplice de la teoría.")

### Caso 10 — Un símplice grande ($4$-símplice)
Prueba más pesada: $2^{5}-1 = 31$ símplices, con la distribución binomial completa.

In [ ]:
from math import comb

S = ComplejoSimplicial([{0, 1, 2, 3, 4}])   # 4-símplice
k = 4
total = len(S.caras())
print("4-símplice: dim", S.dimension(), "| total símplices", total, "(esperado", 2 ** (k + 1) - 1, ")")
reparto = [len(S.caras_de_dimension(l)) for l in range(k + 1)]
binom  = [comb(k + 1, l + 1) for l in range(k + 1)]
print("reparto por dimensión:", reparto)
print("binomiales esperados: ", binom)
assert S.dimension() == 4
assert total == 2 ** (k + 1) - 1
assert reparto == binom
print("Funciona: el 4-símplice genera 31 símplices con el reparto [5,10,10,5,1], todo correcto.")

### Caso 11 — Link de caras internas de un tetraedro
Comprobación estructural fuerte: en un tetraedro sólido, el link de un vértice es la **cara opuesta**, y el link de una arista es la **arista opuesta**.

In [ ]:
T = ComplejoSimplicial([{0, 1, 2, 3}])

lk0 = T.link({0})
print("Lk({0}) =", mostrar(lk0))
assert lk0 == ComplejoSimplicial([{1, 2, 3}]).simplices
print("  -> es el triángulo sólido opuesto {1,2,3} con sus 7 caras.")

lk01 = T.link({0, 1})
print("Lk({0,1}) =", mostrar(lk01))
assert lk01 == ComplejoSimplicial([{2, 3}]).simplices
print("  -> es la arista opuesta {2,3}.")

print("Funciona: los links coinciden con la intuición geométrica de 'lo que rodea' a la cara.")

### Caso 12 — Link del símplice maximal es vacío
No queda ningún símplice disjunto de todos sus vértices.

In [ ]:
T = ComplejoSimplicial([{0, 1, 2, 3}])
print("Lk({0,1,2,3}) =", mostrar(T.link({0, 1, 2, 3})))
assert T.link({0, 1, 2, 3}) == set()
print("Funciona: como el símplice ocupa todos los vértices, nada es disjunto de él; el link es vacío.")

### Caso 13 — Complejo desconexo y vértice aislado
La estrella no debe 'saltar' entre componentes, y un vértice aislado tiene link vacío.

In [ ]:
D = ComplejoSimplicial([{0, 1, 2}, {3, 4, 5}, {6}])  # dos triángulos + vértice suelto
print("dim:", D.dimension(), "| componentes: {0,1,2}, {3,4,5}, {6}")

verts_est0 = set().union(*D.estrella({0}))
print("Vértices que toca St({0}):", sorted(verts_est0))
assert verts_est0 == {0, 1, 2}

print("Lk({6}) =", mostrar(D.link({6})))
assert D.link({6}) == set()
print("Funciona: St({0}) se queda en su triángulo, y el vértice aislado {6} tiene link vacío.")

### Caso 14 — La estrella cerrada es un subcomplejo
Verificación estructural: $\overline{\operatorname{St}}(\tau)$ es cerrada bajo caras (es un subcomplejo de verdad) y contiene a la estrella.

In [ ]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
sc = K.estrella_cerrada({2})

cerrada_bajo_caras = all(frozenset(c) in sc
                         for sigma in sc
                         for r in range(1, len(sigma))
                         for c in combinations(sorted(sigma), r))
assert cerrada_bajo_caras
assert K.estrella({2}) <= sc
print("St_cerrada({2}) =", mostrar(sc))
print("Funciona: es cerrada bajo caras (subcomplejo) y contiene a St({2}), como exige la definición del link.")

---
Si has ejecutado todas las celdas sin que ninguna fallara, cada función está validada: construcción con cierre bajo caras, dimensión, caras (totales y por dimensión), estrella y link, incluyendo casos vacíos, símplices grandes con su combinatoria, links de caras internas, el símplice maximal y complejos desconexos.